In [1]:
import pandas as pd

file_path = '/Users/nitishrmaladakar/Desktop/infosys/infosys-langgraph-email-assistant-group2/data/email_evaluation_dataset_Nitish_R_Maladakar.csv' 
df = pd.read_csv(file_path)
print("Data loaded successfully!")
display(df.head())


Data loaded successfully!


,id,email_text,expected_action,expected_tone
0,1,"Hi team, just a reminder that the Q3 marketing...",respond,neutral
1,2,Invoice #2024-001 is now overdue by 5 days. Pl...,notify,urgent
2,3,Congratulations! You have been shortlisted for...,respond,polite
3,4,"Hey, are we still on for lunch today? Let me k...",respond,casual
4,5,Weekly Newsletter: Top 10 trends in AI this we...,ignore,neutral


In [2]:
df = pd.read_csv(r"/Users/nitishrmaladakar/Desktop/infosys/infosys-langgraph-email-assistant-group2/data/email_evaluation_dataset_Nitish_R_Maladakar.csv")
print(df.head(50))

    id                                         email_text expected_action  \
0    1  Hi team, just a reminder that the Q3 marketing...         respond   
1    2  Invoice #2024-001 is now overdue by 5 days. Pl...          notify   
2    3  Congratulations! You have been shortlisted for...         respond   
3    4  Hey, are we still on for lunch today? Let me k...         respond   
4    5  Weekly Newsletter: Top 10 trends in AI this we...          ignore   
5    6  Urgent: The production server is down. We need...          notify   
6    7  Can you please send me the syllabus for the up...         respond   
7    8  Automatic Reply: I am currently out of the off...          ignore   
8    9  Your subscription to Spotify has been successf...          ignore   
9   10  Attached is the final report for the project. ...         respond   
10  11  Reminder: Your dentist appointment is tomorrow...          notify   
11  12  Please find the attached invoice for the recen...          notify   

In [8]:
import re

def clean_email_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower() 

    text = text.replace('\n', ' ').strip()
    
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    return text

df['Clean_text'] = df['email_text'].apply(clean_email_text)


print("Clean_text column created successfully!")
display(df[['email_text', 'Clean_text']].head())

Clean_text column created successfully!


,email_text,Clean_text
0,"Hi team, just a reminder that the Q3 marketing...",hi team just a reminder that the q marketing s...
1,Invoice #2024-001 is now overdue by 5 days. Pl...,invoice is now overdue by days please proces...
2,Congratulations! You have been shortlisted for...,congratulations you have been shortlisted for ...
3,"Hey, are we still on for lunch today? Let me k...",hey are we still on for lunch today let me kno...
4,Weekly Newsletter: Top 10 trends in AI this we...,weekly newsletter top trends in ai this week ...


In [11]:
def email_assistant(email_text):
    text = email_text.lower()
    
    urgent_keywords = [
        "urgent", "overdue", "security", "breach", "declined", 
        "failed", "immediate", "server", "crash", "alert"
    ]
    ignore_keywords = [
        "newsletter", "automatic reply", "subscription", "deal", 
        "offer", "tracking", "system generated", "no reply", "marketing"
    ]
    if any(word in text for word in urgent_keywords):
        return "notify", "urgent"

    elif any(word in text for word in ignore_keywords):
        return "ignore", "neutral"
    
    elif "thank you" in text or "congratulations" in text:
        return "respond", "polite"

    else :
        return "respond", "neutral"

# Apply assistant function to the dataset

In [19]:
results = df['email_text'].apply(email_assistant)

# Create separate output columns

In [ ]:


df[['predicted_action', 'predicted_tone']] = pd.DataFrame(results.tolist(), index=df.index)


print(df[['email_text', 'predicted_action', 'predicted_tone']].head())

                                          email_text predicted_action  \
0  Hi team, just a reminder that the Q3 marketing...           ignore   
1  Invoice #2024-001 is now overdue by 5 days. Pl...           notify   
2  Congratulations! You have been shortlisted for...          respond   
3  Hey, are we still on for lunch today? Let me k...          respond   
4  Weekly Newsletter: Top 10 trends in AI this we...           ignore   

  predicted_tone  
0        neutral  
1         urgent  
2         polite  
3        neutral  
4        neutral  


# Compare predictions with expected values

In [17]:
df['action_correct'] = df['predicted_action'] == df['expected_action']
df['tone_correct'] = df['predicted_tone'] == df['expected_tone']

# Calculate accuracy 

In [18]:



action_accuracy = df['action_correct'].mean() * 100
tone_accuracy = df['tone_correct'].mean() * 100

print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")


print(df[['expected_action', 'predicted_action', 'action_correct']].head())

Action Accuracy: 57.00%
Tone Accuracy: 70.00%
  expected_action predicted_action  action_correct
0         respond           ignore           False
1          notify           notify            True
2         respond          respond            True
3         respond          respond            True
4          ignore           ignore            True


# Error Analysis

In [ ]:

errors = df[df['action_correct'] == False]

print("Number of Action Errors:", len(errors))
print("\n--- Error Samples ---")
print(errors[['email_text', 'expected_action', 'predicted_action']].head(10))

Number of Action Errors: 43

--- Error Samples ---
                                           email_text expected_action  \
0   Hi team, just a reminder that the Q3 marketing...         respond   
10  Reminder: Your dentist appointment is tomorrow...          notify   
11  Please find the attached invoice for the recen...          notify   
13  Dear Student, the deadline for submitting your...          notify   
16  Meeting canceled: Project Kickoff. We will res...          ignore   
17  I'm running 10 minutes late to the standup. St...          ignore   
19  Happy Birthday! wishing you a fantastic year a...          ignore   
21  Your package has been delivered to the front d...          notify   
27  Don't forget to submit your timesheet for this...          notify   
28  FW: The client wants to change the scope of th...          notify   

   predicted_action  
0            ignore  
10          respond  
11          respond  
13          respond  
16          respond  
17          r

In [ ]:
import os


df.to_csv('../data/milestone2_output_nitish.csv', index=False)

print("File saved successfully.")

File saved successfully.


## `1. Which type of emails were hardest to classify?`

Emails that implied urgency without using specific keywords like 'urgent' or 'deadline' (e.g., 'My screen is black', 'Invoice overdue'). Also, polite emails that required action (e.g., 'Thank you, but can you fix this?') were often misclassified as 'Ignore'.


## `2. Why did your rules fail in some cases?`

The rule-based approach is too rigid . It relies entirely on exact keyword matching. It cannot understand context, synonyms (like 'asap' vs 'urgent'), or the main reason behind a email.


## `3. How could an LLM improve this process?`

An LLM understands real meaning and context. It can detect urgency from tone even if specific keywords are missing and can distinguish between a polite sign-off ('Thank you') and a polite request ('Thank you for doing this, please reply').